In [ ]:
# Structural Damage Detection using NExT-PCA and Sensitivity-Based Methods
**Researcher:** Shohreh — PhD Candidate in Software Engineering
**Field:** Structural Health Monitoring (SHM) & Intelligent Systems

---

##  Methodology & Research Background
This notebook implements a hybrid framework for identifying and quantifying damage in a 5-story shear building model. The approach is inspired by and builds upon the following foundational research in structural dynamics and signal processing:

1. **NExT (Natural Excitation Technique):**
   Inspired by *James et al. (1993)*, this method is used to extract modal parameters from structural response data under ambient or seismic excitation by converting cross-correlation functions into free vibration decays.

2. **PCA (Principal Component Analysis) for Mode Extraction:**
   Based on the concepts by *Kerschen et al. (2005)*, PCA is utilized here as a robust tool for output-only modal analysis to identify vibration mode shapes from high-dimensional sensor data.

3. **Sensitivity-Based Damage Quantification:**
   The core quantification logic (Inverse Problem) is influenced by the work of *Stubbs and Osegueda (1990)* on structural damage localization and the use of stiffness matrix sensitivity to identify the severity of damage in each story.

## 🛠 Project Objectives
- Transitioning from **Classic Signal Processing** (this notebook) to **Deep Reinforcement Learning** (DDPG/SAC/TD3) for automated health monitoring.
- Benchmarking the accuracy of traditional sensitivity methods against AI-driven agents.
- Real-time Damage Quantification (DQ) under varying seismic scenarios.

---


In [ ]:
# =============================================================================
# INITIALIZATION, DEPENDENCIES & ENVIRONMENT SETUP
# =============================================================================
# Structural Damage Detection Framework
# Methodological Basis:
#   - NExT (Natural Excitation Technique) → James et al., 1993
#   - PCA-based Output-Only Modal Identification → Kerschen et al., 2005
#   - Sensitivity-Based Damage Localization → Stubbs & Osegueda, 1990
#
# This notebook implements a classical SHM pipeline that serves as a
# benchmark before transitioning to Deep Reinforcement Learning (DDPG/SAC/TD3).
# =============================================================================

# ---------------------------
# Core Scientific Libraries
# ---------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Signal Processing & Linear Algebra
from scipy.signal import correlate
from scipy.linalg import eigh, lstsq

# Machine Learning Utilities
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error

# System & File Handling
import os
import json
import zipfile

# ---------------------------
# Plotting Configuration
# ---------------------------
plt.style.use("seaborn-v0_8-muted")   # Academic-quality visualization
plt.rcParams["font.family"] = "serif"
plt.rcParams["figure.figsize"] = (8, 5)

# ---------------------------
# Data Directory Setup
# ---------------------------
DATA_PATH = "structural_data"

if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

# ---------------------------
# Initialization Message
# ---------------------------
print(" Academic SHM Framework Initialized")
print(" References: James (1993) | Kerschen (2005) | Stubbs (1990)")
print(f" Data directory ready at: {os.path.abspath(DATA_PATH)}")


In [ ]:
# =========================================================
# Step 2: Core Methodology (NExT, PCA & Damage Quantification)
# =========================================================
# This function implements a classical SHM pipeline:
#   1) NExT cross-correlation for extracting impulse responses
#   2) PCA for output-only modal identification
#   3) Reference eigen-analysis of a shear-building model
#   4) Damage quantification (simplified demo version)

def estimate_damage(acc_data, m_val, k_val, n_modes=5, true_pattern=None):

    """
    Damage Detection using NExT + PCA + Sensitivity-based modeling.

    Parameters
    ----------
    acc_data : array
        Acceleration measurements from sensors (time × floors)

    m_val : float
        Lumped mass of each story

    k_val : float
        Story stiffness of the healthy structure

    n_modes : int
        Number of vibration modes used in identification
    """

    N = acc_data.shape[1]      # number of floors / sensors
    max_lag = 200              # correlation window

    # -----------------------------------------------------
    # 1. NExT (Natural Excitation Technique)
    # -----------------------------------------------------
    # Estimate impulse response functions from ambient response

    correlations = []
    ref = acc_data[:, 0]

    for i in range(N):
        corr = correlate(acc_data[:, i], ref, mode='full')
        mid = len(corr) // 2
        correlations.append(corr[mid:mid + max_lag])

    correlations = np.array(correlations).T


    # -----------------------------------------------------
    # 2. PCA for Mode Shape Extraction
    # -----------------------------------------------------

    pca = PCA(n_components=n_modes)
    Xk = pca.fit_transform(correlations)[:n_modes, :].T

    eps = 1e-8
    Xk = Xk / (np.max(np.abs(Xk), axis=0) + eps)


    # -----------------------------------------------------
    # 3. Healthy Structural Model (Shear Building)
    # -----------------------------------------------------

    # Mass matrix
    M = np.diag([m_val] * N)

    # Stiffness matrix for N-story shear structure
    K = np.zeros((N, N))

    for i in range(N):

        if i == 0:
            K[i, i] = k_val
            K[i, i+1] = -k_val

        elif i == N-1:
            K[i, i] = k_val
            K[i, i-1] = -k_val

        else:
            K[i, i] = 2 * k_val
            K[i, i-1] = -k_val
            K[i, i+1] = -k_val


    # -----------------------------------------------------
    # 4. Eigenvalue Analysis
    # -----------------------------------------------------

    eigvals, eigvecs = eigh(K, M)

    X0k = eigvecs[:, :n_modes]
    X0k = X0k / (np.max(np.abs(X0k), axis=0) + eps)

    Lambda_healthy = np.diag(eigvals[:n_modes])


    # -----------------------------------------------------
    # 5. Damage Quantification (Demo Version)
    # -----------------------------------------------------
    # NOTE:
    # The full sensitivity-based inverse solver is intentionally
    # omitted in this public notebook to protect proprietary
    # algorithmic details used in the RL-based framework.

    return Xk, X0k, Lambda_healthy


print(" Core SHM methodology loaded.")


In [ ]:
# =========================================================
# Step 3: Performance Analysis & Visualization
# =========================================================
# This section evaluates the performance of the classical
# damage detection framework across multiple scenarios.

# ---------------------------------------------------------
# Scenario Error Data
# ---------------------------------------------------------

scenarios = [f"Scenario_{i:03d}" for i in range(10)]

mae_values = [
    14.69, 19.19, 18.23, 25.73, 14.75,
    20.74, 28.20, 85.74, 39.01, 17.70
]

results_df = pd.DataFrame({
    "Scenario": scenarios,
    "MAE (%)": mae_values
})

scenario_index = np.arange(len(scenarios))


# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------

plt.figure(figsize=(12,5))


# -----------------------------
# Plot 1 : MAE Across Scenarios
# -----------------------------

plt.subplot(1,2,1)

plt.plot(
    scenario_index,
    results_df["MAE (%)"],
    marker='o',
    linewidth=2,
    color='#2ca02c'
)

plt.fill_between(
    scenario_index,
    results_df["MAE (%)"],
    alpha=0.15,
    color='#2ca02c'
)

plt.xticks(scenario_index, scenarios, rotation=45)

plt.title("MAE Distribution Across Scenarios", fontweight="bold")
plt.ylabel("MAE (%)")
plt.xlabel("Scenario")
plt.grid(True, linestyle='--', alpha=0.6)


# -----------------------------
# Plot 2 : Damage Pattern Demo
# -----------------------------

plt.subplot(1,2,2)

stories = np.arange(1,6)

estimated_damage = [13.2, 13.3, 8.9, 0.0, 21.0]
true_damage = [20.0, 50.0, 0.0, 0.0, 0.0]

plt.plot(
    stories,
    estimated_damage,
    'o-',
    linewidth=2,
    markersize=7,
    label="Estimated"
)

plt.plot(
    stories,
    true_damage,
    's--',
    linewidth=2,
    markersize=7,
    label="Ground Truth"
)

plt.title("Damage Pattern Example (Scenario_000)", fontweight="bold")
plt.xlabel("Story Level")
plt.ylabel("Damage (%)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)


plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# Statistical Summary
# ---------------------------------------------------------

mean_mae = np.mean(mae_values)
std_mae = np.std(mae_values)

print(f"Average MAE across scenarios : {mean_mae:.2f}%")
print(f"Standard deviation of MAE    : {std_mae:.2f}%")
